In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('cars.csv')
df.sample(2)

,brand,km_driven,fuel,owner,selling_price
5005,Fiat,35000,Petrol,First Owner,80000
6079,Hyundai,31377,Petrol,Second Owner,320000


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   brand          8128 non-null   str  
 1   km_driven      8128 non-null   int64
 2   fuel           8128 non-null   str  
 3   owner          8128 non-null   str  
 4   selling_price  8128 non-null   int64
dtypes: int64(2), str(3)
memory usage: 317.6 KB


In [4]:
df.shape

(8128, 5)

In [5]:
df.nunique(dropna=False)

brand             32
km_driven        921
fuel               4
owner              5
selling_price    677
dtype: int64

In [6]:
df['owner'].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

In [7]:
?pd.get_dummies

Signature:
pd.get_dummies(
    data,
    prefix=None,
    prefix_sep: 'str | Iterable[str] | dict[str, str]' = '_',
    dummy_na: 'bool' = False,
    columns=None,
    sparse: 'bool' = False,
    drop_first: 'bool' = False,
    dtype: 'NpDtype | None' = None,
) -> 'DataFrame'
Docstring:
Convert categorical variable into dummy/indicator variables.

Each variable is converted in as many 0/1 variables as there are different
values. Columns in the output are each named after a value; if the input is
a DataFrame, the name of the original variable is prepended to the value.

Parameters
----------
data : array-like, Series, or DataFrame
    Data of which to get dummy indicators.
prefix : str, list of str, or dict of str, default None
    A string to be prepended to DataFrame column names.
    Pass a list with length equal to the number of columns
    when calling get_dummies on a DataFrame. Alternatively, `prefix`
    can be a dictionary mapping column names to prefixes.
prefix_sep : str, lis

In [8]:
pd.get_dummies(df,columns=['fuel','owner'],dtype=int)

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,0,1,0,0,1,0,0,0,0
1,Skoda,120000,370000,0,1,0,0,0,0,1,0,0
2,Honda,140000,158000,0,0,0,1,0,0,0,0,1
3,Hyundai,127000,225000,0,1,0,0,1,0,0,0,0
4,Maruti,120000,130000,0,0,0,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,0,1,1,0,0,0,0
8124,Hyundai,119000,135000,0,1,0,0,0,1,0,0,0
8125,Maruti,120000,382000,0,1,0,0,1,0,0,0,0
8126,Tata,25000,290000,0,1,0,0,1,0,0,0,0


In [9]:
# solving multicollinearity problem
pd.get_dummies(df,columns=['fuel','owner'],drop_first=True,dtype=int)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,1,0,0,0,0,0,0
1,Skoda,120000,370000,1,0,0,0,1,0,0
2,Honda,140000,158000,0,0,1,0,0,0,1
3,Hyundai,127000,225000,1,0,0,0,0,0,0
4,Maruti,120000,130000,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,1,0,0,0,0
8124,Hyundai,119000,135000,1,0,0,1,0,0,0
8125,Maruti,120000,382000,1,0,0,0,0,0,0
8126,Tata,25000,290000,1,0,0,0,0,0,0


In [10]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(df.iloc[:,0:4],df.iloc[:,-1],test_size=0.2,random_state=0)
X_train.sample(2)

,brand,km_driven,fuel,owner
758,Hyundai,80000,Petrol,Second Owner
1250,Maruti,35000,Diesel,First Owner


In [11]:
X_train.shape

(6502, 4)

In [12]:
X_test.shape

(1626, 4)

In [33]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first',sparse_output=False,dtype=int)

In [26]:
?ohe

Type:        OneHotEncoder
String form: OneHotEncoder(drop='first')
File:        c:\users\anjli kumari\anaconda3\envs\machine_learning\lib\site-packages\sklearn\preprocessing\_encoders.py
Docstring:  
Encode categorical features as a one-hot numeric array.

The input to this transformer should be an array-like of integers or
strings, denoting the values taken on by categorical (discrete) features.
The features are encoded using a one-hot (aka 'one-of-K' or 'dummy')
encoding scheme. This creates a binary column for each category and
returns a sparse matrix or dense array (depending on the ``sparse_output``
parameter).

By default, the encoder derives the categories based on the unique values
in each feature. Alternatively, you can also specify the `categories`
manually.

This encoding is needed for feeding categorical data to many scikit-learn
estimators, notably linear models and SVMs with the standard kernels.

Note: a one-hot encoding of y labels should use a LabelBinarizer
instead.


In [34]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']])

In [35]:
X_test_new = ohe.fit_transform(X_test[['fuel','owner']])

In [36]:
np.hstack((X_train[['brand','km_driven']].values,X_train_new)).shape

(6502, 9)

In [37]:
np.hstack((X_test[['brand','km_driven']].values,X_test_new)).shape

(1626, 9)

In [38]:
cnts = df['brand'].value_counts()
cnts.sample(2)

brand
Toyota    488
BMW       120
Name: count, dtype: int64

In [40]:
mn = 100
repl = cnts[cnts <= mn].index

In [43]:
pd.get_dummies(df['brand'].replace(repl,'Others'),dtype=int)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Others,Renault,Skoda,Tata,Toyota,Volkswagen
0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8123,0,0,0,0,1,0,0,0,0,0,0,0,0
8124,0,0,0,0,1,0,0,0,0,0,0,0,0
8125,0,0,0,0,0,0,1,0,0,0,0,0,0
8126,0,0,0,0,0,0,0,0,0,0,1,0,0
